# 1. What you'll learn

        - construct normalized adjacency message passing
- train a two-layer GCN without a graph framework
- compare honestly with graph-feature LightGBM

        **The one question this notebook answers:** When does message passing help node classification, and can engineered graph features plus LightGBM win?

# 2. Setup

This lesson keeps graph neural networks versus graph features small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
from torch import nn
import networkx as nx
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A seeded 500-node transaction graph has four communities, node risk features, homophilous edges, and fraud labels influenced by both node and neighbourhood risk.

**Small example:** If a node has feature 2 and neighbours with [1,4], mean aggregation including itself gives (2+1+4)/3=7/3; a learned weight then transforms that local context.

In [ ]:
rng=np.random.default_rng(SEED);n=500;community=np.repeat(np.arange(4),125);G=nx.stochastic_block_model([125]*4,np.full((4,4),.008)+np.eye(4)*.045,seed=SEED);risk=rng.normal(size=(n,4));A=nx.to_numpy_array(G);neigh=(A@risk[:,0])/np.maximum(A.sum(1),1);logit=-2+.8*risk[:,0]+.7*neigh+.45*(community==3);prob=1/(1+np.exp(-logit));y=rng.binomial(1,prob);print("Nodes/edges:",n,G.number_of_edges(),"fraud share:",round(y.mean(),3),"features:",risk.shape)

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
degree=A.sum(1);fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].hist(degree,bins=25,color="#176b66");axes[0].set_title("Node degree")
axes[1].hist(risk[y==0,0],bins=25,alpha=.6,label="normal");axes[1].hist(risk[y==1,0],bins=25,alpha=.6,label="fraud");axes[1].legend();axes[1].set_title("Node risk overlaps")
axes[2].scatter(risk[:,0],neigh,c=y,cmap="coolwarm",s=12);axes[2].set(title="Node + neighbour signal",xlabel="own risk",ylabel="neighbour risk");plt.tight_layout();plt.show()

**Takeaway:** Degree varies enough that raw sums would overweight hubs.

**Takeaway:** Own risk is predictive but cannot separate all fraud.

**Takeaway:** Neighbour context contains complementary signal, motivating either message passing or engineered aggregation.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
idx=np.arange(n);train_idx,test_idx=train_test_split(idx,test_size=.3,stratify=y,random_state=SEED);train_idx,val_idx=train_test_split(train_idx,test_size=.25,stratify=y[train_idx],random_state=SEED);I=np.eye(n);At=A+I;deg=np.sum(At,1);An=At/np.sqrt(deg[:,None]*deg[None,:]);X=torch.tensor(risk,dtype=torch.float32);Adj=torch.tensor(An,dtype=torch.float32);target=torch.tensor(y,dtype=torch.float32);print("Train/val/test:",len(train_idx),len(val_idx),len(test_idx),"adjacency:",Adj.shape)

# 6. Train

        Training turns the assumptions behind graph neural networks versus graph features into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Add self-loops so a node retains its own features.
2. Normalize adjacency by both endpoint degrees.
3. Aggregate neighbours, transform features, and apply ReLU.
4. Repeat once, then optimize labelled training nodes only.

In [ ]:
torch.manual_seed(SEED)
class GCN(nn.Module):
 def __init__(self): super().__init__();self.w1=nn.Linear(4,16,bias=False);self.w2=nn.Linear(16,1,bias=False)
 def forward(self,x,adj): return self.w2(adj@torch.relu(self.w1(adj@x))).squeeze(1)
gcn=GCN();opt=torch.optim.Adam(gcn.parameters(),lr=.025,weight_decay=1e-3);history=[]
for epoch in range(180):
 opt.zero_grad();score=gcn(X,Adj);loss=nn.functional.binary_cross_entropy_with_logits(score[train_idx],target[train_idx]);loss.backward();opt.step()
 with torch.no_grad(): history.append(roc_auc_score(y[val_idx],score[val_idx].numpy()))
with torch.no_grad(): gprob=torch.sigmoid(gcn(X,Adj)).numpy()
engineered=np.c_[risk,degree,neigh,A@(risk[:,1])/np.maximum(degree,1)];tree=LGBMClassifier(n_estimators=140,num_leaves=15,verbosity=-1,random_state=SEED,n_jobs=1).fit(engineered[train_idx],y[train_idx]);tprob=tree.predict_proba(engineered)[:,1];print("Best validation GCN AUC:",round(max(history),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
ga=roc_auc_score(y[test_idx],gprob[test_idx]);ta=roc_auc_score(y[test_idx],tprob[test_idx]);base=.5;fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(history);axes[0].axhline(.5,ls="--",color="black");axes[0].set(title="GCN validation AUC",xlabel="epoch")
axes[1].bar(["random","GCN","graph features + LightGBM"],[base,ga,ta],color=[".6","#176b66","#d38b45"]);axes[1].set_ylim(.45,1);axes[1].set_title("Held-out node AUC")
axes[2].scatter(tprob[test_idx],gprob[test_idx],c=y[test_idx],cmap="coolwarm",s=14);axes[2].set(title="Models disagree",xlabel="LightGBM probability",ylabel="GCN probability");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"GCN AUC={ga:.3f}; graph-feature LightGBM={ta:.3f}. On tabular fraud graphs, the simpler model often wins.")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Message passing trusts graph structure; rewired, adversarial, or future-derived edges can damage or leak predictions. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
perm=rng.permutation(n);rewired=A[perm][:,perm]+np.eye(n);rd=rewired.sum(1);rewired=rewired/np.sqrt(rd[:,None]*rd[None,:])
with torch.no_grad(): badprob=torch.sigmoid(gcn(X,torch.tensor(rewired,dtype=torch.float32))).numpy()
print(f"Correct-graph GCN AUC={ga:.3f}; randomly rewired inference graph AUC={roc_auc_score(y[test_idx],badprob[test_idx]):.3f}; LightGBM graph-feature AUC={ta:.3f}")

**Use this when…** edges carry predictive relational information unavailable in node rows and graph construction is trustworthy.

        **Don't use this when…** edges are noisy/leaky/dynamic, engineered aggregates already win, or deployment cannot serve neighbourhoods.

        ## Try this

        1. Remove neighbour-driven label signal and compare models.
2. Add a third GCN layer and inspect oversmoothing.
3. Train graph-feature LightGBM without neighbour aggregates.